# 01 · Data and feature store

Stages raw data and the Phase-2 / Phase-4 feature caches from Drive, reloads the Phase-5 context (nothing from Phases 1-4 is recomputed), inspects the feature inventory and builds the **missing** Phase-5 structural-profile features (Experiment D). New feature files are written to `data/feature_store/phase5/` and pushed to Drive.

CPU-heavy (profile features: ~30 s per field and scale); no GPU needed.

In [ ]:
# --- session bootstrap (idempotent; identical in notebooks 00-07) ---------------------------------
REPO = 'mybenkhadda/GEMS---drivendata'
COMMIT = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/GEMS_2'
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
url = f'https://github.com/{REPO}.git'   # token-free remote: a token never goes into the URL / .git/config
# private fork only: GITHUB_TOKEN (Colab Secret) is sent as an HTTP header through env vars, per command
genv = {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + __import__('base64').b64encode(f'x-access-token:{token}'.encode()).decode()} if token else None
if not os.path.exists('/content/GEMS/.git'):
    subprocess.run(['git', 'clone', '-q', url, '/content/GEMS'], check=True, env=genv)
subprocess.run(['git', '-C', '/content/GEMS', 'remote', 'set-url', 'origin', url], check=True)  # drop an old embedded token
subprocess.run(['git', '-C', '/content/GEMS', 'fetch', '-q', 'origin'], check=True, env=genv)
subprocess.run(['git', '-C', '/content/GEMS', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/GEMS[colab]'], check=True)
if '/content/GEMS' not in sys.path:
    sys.path.insert(0, '/content/GEMS')
from gems_bench.runtime import colab
colab.configure(drive_root=DRIVE_ROOT)
colab.stage_core();

In [ ]:
# Stage the Phase-2 / Phase-4 feature caches and the legacy model maps (skip-unchanged)
st = colab.stage_legacy()
st.groupby('action').bytes.agg(['count', 'sum'])

In [ ]:
from gems_bench.phase5_context import load_context5
from gems_bench.models.zoo import register_aliases
ctx = load_context5()
register_aliases(ctx)
display(ctx.data.summary())
print('features in store:', len(ctx.store.specs), '| cached:', len(ctx.store.available))
print('Phase-5 profile features:', len([n for n in ctx.store.p5_names if n.startswith('p5:')]))
print('models with OOF maps:', ctx.model_names())

In [ ]:
# Build missing profile features (label-free; orientation from measured fields only) and push them to Drive
names = [n for n in ctx.store.p5_names if n.startswith('p5:')]
reg = ctx.store.build(names)
colab.sync().push(['data/feature_store/phase5'])
reg[['name', 'family', 'scale_km', 'support_km', 'cached']]

In [ ]:
# Feature-engineering atlas: profile responses around mapped faults (regenerable from the cache)
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from gems_bench.visualization.figures import raster_windows
rr, cc = np.nonzero(ctx.data.labels)
idx = np.random.default_rng(2026).choice(len(rr), 3, replace=False)
wins = [(int(rr[i]), int(cc[i]), 60) for i in idx]
show = ['raw:det_elev', 'p5:terrain:profile_step@0.5km', 'p5:terrain:profile_asymmetry@0.5km',
        'p5:terrain:profile_offset@0.5km', 'p5:magnetic:profile_step@1km']
fig = raster_windows({n.split(':', 1)[1]: np.asarray(ctx.store.get(n)) for n in show if ctx.store.has(n)}, wins,
                     ctx.data.labels, title='Fault-normal profile features (mapped faults in black)')
out = Path('/content/gems_work/reports/figures'); out.mkdir(parents=True, exist_ok=True)
for ext in ('png', 'svg'):
    fig.savefig(out / f'profile_feature_atlas.{ext}', dpi=200, bbox_inches='tight')
colab.sync().push(['reports/figures'])